## Nestlé GPT Basics

### Authentication

NesGen Accelerator uses one-way authentication. This is based on the credentials provided through specific HTTP headers:

| Header | Description |
| --- | --- |
| `client_id` | Client ID is a unique identifier assigned to the client application. |
| `client_secret` | Client Secret is a secret known only to the application and the authorization server. It is assigned to the `client_id` parameter and attached to a particular client application. |

Client ID and Client Secret are the credentials provided after requesting access through this API Catalog instance. Once access is granted, the credentials can be accessed in the "My Applications" section.

In [1]:
import sys
from pathlib import Path

import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Shared Nestlé credentials / URL from project-root config (.env)
sys.path.insert(0, str(Path("..").resolve()))
from config import auth_headers, build_openai_url, deploymentName

headers = auth_headers


#### Provider-Specific APIs

Provider-specific APIs follow the endpoint structure:
`{OPENAI_API_BASE_URL}/deployments/{deploymentName}/{action}/{action_extension}`

The base URL and credentials live in `.env` and are loaded via `config.py` — do not hardcode them.

The available models in the provider-specific APIs, as well as the path variables to direct queries to each model, are described in the following table:

| Provider | Model | Context | deploymentName | action | action-extension |
| --- | --- | --- | --- | --- | --- |
| MS Azure | OpenAI GPT-3.5 | 8k | ChatGPT | chat | completions |
| MS Azure | OpenAI GPT-3.5 | 16k | ChatGPTv16k | chat | completions |
| MS Azure | OpenAI GPT-4 | 8k | GPT-4 | chat | completions |
| MS Azure | OpenAI GPT-4 | 32k | GPT-4v32k | chat | completions |
| MS Azure | OpenAI text-embedding-ada-002 | NA | AzEmbeddings | embeddings | NA |
| MS Azure | OpenAI Whisper | NA | Whisper | audio | transcriptions |


### Text completition

In [2]:
url = build_openai_url(deploymentName, "chat", "completions")


In [3]:
data = {
    "messages": [{
            "role": "system",
            "content": "You are the best NER in the world. You are able to extract any entity from any text. Take your time to solve the problem."
        },
        {
            "role": "user",
            "content": """Sentence: The authorities seized 53 600 litres of wine and 30 tons of apples with several irregularities in the documentation (total value of 100 000 Euros).
Instruct: Extract entities for each label. Reply only with a json file with the following format: {"Food":<Food_list>,"Quantity":<Quantity_list>,"Location":<Location_list>}. If there is no entity, leave it as an empty array. 
Output:"""
        }
    ]
}  
req = requests.post(url, headers=headers, json=data, verify=False)

res = req.json()
res["choices"][0]["message"]

{'role': 'assistant',
 'content': '{\n  "Food": ["wine", "apples"],\n  "Quantity": ["53 600 litres", "30 tons"],\n  "Location": []\n}'}

In [23]:
# Synthetic data
label = "food"
entity = "apple"
instruction = f'Please generate a synthetic dataset in JSON format that includes a list of food entities. Each entity should represent a type of food, and the standard version for all entities should be labeled as "apple".'+' The dataset structure should follow the format: {"label":"","entities":[]}. Feel free to be creative with the entities you generate.'
example = '{"label":"wheat","entities":["wheat","wheat grain","wheat flour","wheat grain with amaranth","grano","triticum"]}.'
system = ""
sentence = ""

data = {
    "messages": [{
            "role": "system",
            "content": "Take your time to solve the problem."
        },
        {
            "role": "user",
            "content": f"""Instruct: {instruction} 
Example: {example}
Output:"""
        }
    ]
} 

req = requests.post(url, headers=headers, json=data, verify=False)

res = req.json()
res["choices"][0]["message"]


### Embeddings

In [4]:
url = build_openai_url("AzEmbeddings", "embeddings")


In [5]:
data = {
    "input": "I like to eat apples and bananas.",
}  

req = requests.post(url, headers=headers, json=data, verify=False)

res = req.json()
res

{'object': 'list',
 'data': [{'object': 'embedding',
   'index': 0,
   'embedding': [-0.004665158,
    -0.019361973,
    0.012749346,
    -0.0039794743,
    0.0021321967,
    0.0029352927,
    -0.017370671,
    -0.026726035,
    0.010632804,
    -0.012066793,
    0.0003471473,
    0.0065687937,
    0.005729691,
    -0.01062028,
    -0.01449017,
    0.017408242,
    0.037371363,
    -0.010783091,
    -0.0060083484,
    -0.018748302,
    -0.009887631,
    0.0055105225,
    0.027953379,
    -0.025385976,
    -0.0086853355,
    0.011027307,
    0.017057573,
    -0.019612452,
    0.020914938,
    0.004195512,
    0.04719011,
    -0.012642892,
    0.0078462325,
    -0.0033219685,
    0.0015842753,
    -0.011597145,
    -0.014715601,
    0.0016907286,
    -0.003057401,
    -0.023657676,
    -0.023519913,
    0.020626888,
    -0.0027443029,
    0.0029431202,
    -0.017007478,
    0.02805357,
    -0.0052537825,
    -0.0108457105,
    -0.020063313,
    0.0030981035,
    0.012605321,
    -0.02472

## RAG

In [ ]:
from scripts.rag import RAG

rag_system = RAG("test")

In [ ]:
from tqdm import tqdm

# lis of sentences about different food
sentences = [
  "1. Pizza: A popular Italian dish made with a thin, round crust topped with tomato sauce, cheese, and various toppings such as pepperoni, mushrooms, and vegetables.",
  "2. Sushi: A traditional Japanese dish consisting of vinegared rice combined with raw or cooked seafood, vegetables, and sometimes wrapped in seaweed.",
  "3. Tacos: A Mexican dish made with a tortilla filled with various ingredients such as seasoned meat, beans, cheese, lettuce, and salsa.",
  "4. Pasta: A staple Italian food made from unleavened dough of wheat flour, water, and sometimes eggs, shaped into various forms like spaghetti, penne, or lasagna.",
  "5. Burgers: A popular fast food item consisting of a grilled or fried patty made from ground meat, typically beef, served in a bun with toppings like lettuce, tomato, and cheese.",
  "6. Sushi: A traditional Japanese dish consisting of vinegared rice combined with raw or cooked seafood, vegetables, and sometimes wrapped in seaweed.",
  "7. Curry: A dish originating from the Indian subcontinent, made with a combination of spices, meat or vegetables, and a sauce served with rice or bread.",
  "8. Salad: A dish made with a mixture of raw or cooked vegetables, fruits, and sometimes meat or cheese, typically dressed with a vinaigrette or creamy dressing.",
  "9. Tiramisu: An Italian dessert made with layers of coffee-soaked ladyfingers, mascarpone cheese, and cocoa, often dusted with chocolate shavings.",
  "10. Sushi: A traditional Japanese dish consisting of vinegared rice combined with raw or cooked seafood, vegetables, and sometimes wrapped in seaweed."
]

for sentence in tqdm(sentences):
  rag_system.add_document(sentence)

In [ ]:
rag_system.search("apple")

In [ ]:
import pandas as pd

